# Forecast walkthrough

Runs the forecast pipeline of the app one step at a time and prints what happens at each step.
Every cell calls the app's own functions from `backend/app/`, so what you see is what the app does.

The flow (it follows `run_forecast_pipeline` in `backend/app/pipeline.py`):

1. Load a profile. From it the app takes the sign to read for and works out your **priorities**
   in code: one short text, the only description of you that an LLM gets.
2. Pick the sign and month, and select the sources.
3. Read the source sites directly (free). This is where the forecast texts come from.
4. Trim the texts to the prompt budget.
5. **Extract**: one LLM call per source text, with nothing about you in it. The Extractor lists the
   dated events of that text, each with a quote. The answers are cached, so a second run for the
   same texts is free. **A call for a text that is not cached costs money** and only runs with
   `RUN_LLM = True`.
6. **Verify** in code: an event is kept only if its quote is in the source text and names the day.
7. **Rank** in code: your focus areas first, then what your life context points to, then the rest.
8. **Synthesize**: one LLM call that gets the verified events (no source text) and your priorities,
   and only words the forecast. **This call costs money** and only runs with `RUN_LLM = True`.
9. Assemble in code: dates, types, life areas and citations come from the verified events; the
   `.ics` calendar is built from the result.

**Tavily** (web search) is not part of a normal forecast and is not called here. The app uses it
only when fewer than 2 default sources are readable, or for a bare site you added yourself. Step 3
tells you whether the app would have fallen back to it.

**Kernel:** pick the Python interpreter in `backend/.venv` (create it with `cd backend && uv sync`).

**Privacy:** once run, the outputs hold your name, birth date and life context. Clear all outputs
before committing this notebook.

## Settings

In [ ]:
# A file name in data/profiles/, or None for the most recently saved profile.
PROFILE_FILE = None
# e.g. "December 2026", or None for the month stored in the profile.
TARGET_MONTH = None
# The life context is never stored on the server, so type it here if you want it in the prompt
# (personal profiles only). Example:
# LIFE_CONTEXT = {
#     "relationship": "married",        # single, dating, inRelationship, married, separated, complicated
#     "work": "consideringChange",      # employed, jobSeeking, consideringChange, selfEmployed, ...
#     "kids": ["under5"],
#     "goals": "Decide about the job offer",
#     "upcomingEvents": [{"date": "2026-11-20", "label": "Final interview"}],
# }
LIFE_CONTEXT = {
    "relationship": "married",  # single, dating, inRelationship, married, separated, complicated
    "work": "consideringChange",  # employed, jobSeeking, consideringChange, selfEmployed, ...
    "kids": ["under5"],
    "goals": "Decide about the job offer",
    "upcomingEvents": [{"date": "2026-11-20", "label": "Final interview"}],
}
# Default sources to leave out: rux-youtube, cafe-astrology, astrology-zone, astrostyle, sally-kirkman
DISABLED_SOURCES = []
# True: use data/cache/ like the app (source texts for 7 days, extractions until a text changes).
# False: read every site again and extract every text again (a temporary cache is used; the app's
# cache is not touched).
USE_APP_CACHE = True
# True makes the paid Nebius calls: one per source text that is not in the extraction cache, and
# one for the Synthesizer. False uses cached extractions and the Synthesizer answer of the last paid
# run, if there are any.
RUN_LLM = False
# Longest text printed per item, or None to print everything.
MAX_CHARS = None

## Setup

In [ ]:
import sys
from pathlib import Path

BACKEND = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "app" / "pipeline.py").is_file()
)
REPO_ROOT = BACKEND.parent
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

In [ ]:
import json
import os
import tempfile
import textwrap
import time

import httpx
from dotenv import load_dotenv
from pydantic import ValidationError

from app.config import PipelineDeps, PipelineError, load_agent_config
from app.extract import (
    EXTRACTOR_PROMPT,
    _cache_path,
    _read_cache,
    extract_document,
    extractor_payload,
    verify_answer,
)
from app.forecast import assemble_forecast, rank_events, synthesizer_payload
from app.llm import call_nebius_chat
from app.models import PersonalProfile, error_messages, parse_profile, stored_profile
from app.pipeline import MIN_CURATED_SOURCES, reading_sign
from app.priorities import build_priorities
from app.prompts import SYNTHESIZER_PROMPT
from app.sources import (
    MAX_TOTAL_DOCUMENT_LENGTH,
    gather_documents,
    select_sources,
    trim_documents,
    validate_sources_config,
)
from app.validation import MONTH_NAMES, parse_month_range, primary_sign
from app.webpages import Target

load_dotenv(REPO_ROOT / ".env")
APP_DATA_DIR = Path(os.environ.get("DATA_DIR") or REPO_ROOT / "data")
ANSWER_FILE = APP_DATA_DIR / "notebook" / "last_answer.json"

http_log = []  # every HTTP request the pipeline makes, in order


class RecordingTransport(httpx.AsyncBaseTransport):
    """Passes requests through unchanged and notes them. Headers (the API key) are not kept."""

    def __init__(self):
        self.inner = httpx.AsyncHTTPTransport()

    async def handle_async_request(self, request):
        entry = {"method": request.method, "url": str(request.url), "status": None}
        http_log.append(entry)
        if request.method == "POST":
            entry["request"] = json.loads(request.content) if request.content else None
        response = await self.inner.handle_async_request(request)
        entry["status"] = response.status_code
        if request.method == "POST":
            await response.aread()
            try:
                entry["response"] = response.json()
            except ValueError:
                entry["response"] = response.text
        return response

    async def aclose(self):
        await self.inner.aclose()


http = httpx.AsyncClient(transport=RecordingTransport(), timeout=httpx.Timeout(170.0, connect=10.0))
data_dir = APP_DATA_DIR if USE_APP_CACHE else Path(tempfile.mkdtemp(prefix="moroscope-notebook-"))
deps = PipelineDeps(env=os.environ, http=http, data_dir=data_dir, allowed_hosts=frozenset())


def show(text, indent=""):
    """Print a text, wrapped for reading (only the line breaks differ from the original)."""
    text = str(text)
    if MAX_CHARS is not None and len(text) > MAX_CHARS:
        text = text[:MAX_CHARS] + f" ... [{len(text) - MAX_CHARS} more characters, raise MAX_CHARS]"
    for paragraph in text.splitlines() or [""]:
        print(
            textwrap.fill(
                paragraph,
                100,
                initial_indent=indent,
                subsequent_indent=indent,
                break_long_words=False,
                break_on_hyphens=False,
            )
        )


def show_json(value):
    show(json.dumps(value, indent=2, ensure_ascii=False))


print("Model:", os.environ.get("NEBIUS_MODEL") or "(NEBIUS_MODEL is not set)")
print("Cache:", data_dir / "cache")
print("Paid LLM calls:", "ON" if RUN_LLM else "off")

## 1. The profile, and what an LLM learns about you

The stored profile is everything the server keeps. From it the app takes the sign the sources are
read for. Your focus areas and life context are turned into **priorities** in code
(`backend/app/priorities.py`): each answer that points to a life area becomes a priority with its
houses, and the text below is built from a fixed template. That text, and the dated plans you
entered, are all the Synthesizer gets about you. The Extractor gets nothing. Never the name or the
birth date, time and place.

In [ ]:
profiles_dir = APP_DATA_DIR / "profiles"
profile_path = (
    profiles_dir / PROFILE_FILE
    if PROFILE_FILE
    else max(profiles_dir.glob("*.json"), key=lambda p: p.stat().st_mtime)
)
raw_profile = json.loads(profile_path.read_text(encoding="utf-8"))
if TARGET_MONTH:
    raw_profile["targetMonth"] = TARGET_MONTH
if LIFE_CONTEXT:
    raw_profile["lifeContext"] = LIFE_CONTEXT
if "focusAreas" not in raw_profile and raw_profile.get("priorities"):
    print("Note: this profile was saved before the life areas existed. Its old focus topics are")
    print("ignored here (the app converts them in the browser); save the profile again to fix.\n")

try:
    profile = parse_profile(raw_profile)
except ValidationError as exc:
    raise AssertionError(f"The profile is not valid: {error_messages(exc)}") from None
is_personal = isinstance(profile, PersonalProfile)
if LIFE_CONTEXT and not is_personal:
    print("Note: this is a general profile, so the app ignores LIFE_CONTEXT.\n")

# What the app stores (never the life context), read back the way the pipeline does.
snapshot = parse_profile(stored_profile(profile))
life_context = profile.life_answers() if is_personal else {}
reading = reading_sign(snapshot)
priorities = build_priorities(reading["focusAreas"], life_context)
upcoming_events = life_context.get("upcomingEvents") or []

print(f"Stored profile ({profile_path.name}):")
show_json(stored_profile(snapshot))
print(f"\nRead for: {reading['sign']} ({reading['signKind']} sign)")
print("\nPriorities, in order:")
for number, priority in enumerate(priorities.ranked, 1):
    origin = "picked as a focus area" if priority.picked else "from the life context"
    houses = ", ".join(str(house) for house in priority.houses)
    print(f"  {number}. {priority.area:<14} houses {houses:<8} {origin}")
if not priorities.ranked:
    print("  (none)")
print("\nThe text the Synthesizer gets about you:")
show(priorities.narrative(), indent="  ")
if upcoming_events:
    print("\nYour dated plans (also sent to the Synthesizer):")
    show_json(upcoming_events)

## 2. Sign, month and sources

The sources are read for one sign: the Rising sign when it leads, otherwise the Sun sign (in general
mode the chosen sign).

In [ ]:
month_range = parse_month_range(snapshot.target_month)
target = Target(reading["sign"], MONTH_NAMES[month_range.month - 1], month_range.year)
month = {
    "firstDate": month_range.first_date,
    "lastDate": month_range.last_date,
    "label": target.label,
}

source_errors, sources_config = validate_sources_config({"disabledCurated": DISABLED_SOURCES})
assert not source_errors, source_errors
exclusive, selected = select_sources(sources_config)

print(f"Reading for: {target.sign} ({primary_sign(snapshot)[1]} sign), {target.label}")
print("Sources:")
for source in selected:
    print(f"  {source.id:<16} {source.name:<30} {source.url}")

## 3. Read the source sites (free)

Each source has its own reader (`backend/app/webpages.py`, `youtube.py`) that checks the month and
year and cuts out the part for the sign. A source that cannot be read is reported as `failed` or
`no_forecast`; the app then builds the forecast from the others.

In [ ]:
before = len(http_log)
started = time.time()
gathered = await gather_documents(deps, selected, target)

print(f"Read in {time.time() - started:.1f} s\n")
for entry in gathered.report:
    print(
        f"  {entry['status']:<12} {entry['name']:<30} {entry['documents']} document(s)  {entry.get('reason', '')}"
    )

requests_made = http_log[before:]
print(
    "\nPages requested:"
    if requests_made
    else "\nNo page was requested: every source came from the cache."
)
for entry in requests_made:
    print(f"  {entry['status']}  {entry['url']}")
if requests_made:
    print("  (the YouTube transcript download and sources answered from the cache are not listed)")

used = sum(entry["status"] == "used" for entry in gathered.report)
print()
if used < MIN_CURATED_SOURCES:
    print(
        f"Only {used} source(s) readable. The app would now ask the LLM for a search query and "
        "search the web with Tavily. This notebook does not do that."
    )
else:
    print(f"{used} sources readable: the app does not call Tavily for this forecast.")

### The source texts

This is the raw material: what each site says for this sign and month, as text. Everything in the
forecast has to come from here.

In [ ]:
for number, doc in enumerate(gathered.documents, 1):
    print("=" * 100)
    print(f"Document {number}: {doc['source']} | {doc['title']}")
    print(f"{doc['url']}  ({len(doc['content']):,} characters)")
    print("=" * 100)
    show(doc["content"])
    print()

## 4. Prompt budget

The texts together may take up to 90,000 characters. Over that, the longest ones are cut at the end.
Each text is sent to the Extractor in a call of its own.

In [ ]:
documents = trim_documents(gathered.documents)
kept = {doc["url"]: len(doc["content"]) for doc in documents}

for doc in gathered.documents:
    size, after = len(doc["content"]), kept.get(doc["url"])
    change = "dropped" if after is None else "unchanged" if after == size else f"cut to {after:,}"
    print(f"  {size:>7,} characters  {change:<16} {doc['source']}: {doc['title'][:50]}")
print(
    f"\nTotal to extract from: {sum(kept.values()):,} of at most {MAX_TOTAL_DOCUMENT_LENGTH:,} characters"
)

## 5. Extract: one LLM call per source text

Each call has two messages: the system prompt (the rules, printed once below) and a user message
with the month, the sign and **one** source text. Nothing about you. The answer is cached under
`data/cache/extractions/`, keyed by the text, sign, month, model and prompt, so a text that was
extracted before costs nothing. A text that is not cached is sent only with `RUN_LLM = True`.

In [ ]:
config = load_agent_config(os.environ)
print("SYSTEM PROMPT (the same for every document)")
print("-" * 100)
show(EXTRACTOR_PROMPT)
print()

raw_extractions = {}  # document URL -> the events as the Extractor returned them, unchecked
for doc in documents:
    payload = extractor_payload(month, reading["sign"], doc)
    path = _cache_path(deps, config, payload)
    cached = _read_cache(path)
    sent = len(json.dumps(payload, ensure_ascii=False, separators=(",", ":")))
    print(f"{doc['source']}: user message of {sent:,} characters (month, sign, document)")
    if cached is not None:
        raw_extractions[doc["url"]] = cached
        print(f"  from the cache: {len(cached)} event(s) returned earlier, no call made")
    elif RUN_LLM:
        before = len(http_log)
        started = time.time()
        result = await extract_document(config, deps, month, month_range, reading["sign"], doc)
        calls = [entry for entry in http_log[before:] if entry["method"] == "POST"]
        print(
            f"  {len(calls)} request(s) to {config.nebius_model} in {time.time() - started:.1f} s"
        )
        for entry in calls:
            body = entry.get("response")
            print("  Token usage:", body.get("usage") if isinstance(body, dict) else None)
        if result.error:
            print(f"  The extraction failed: {result.error.message}")
        else:
            raw_extractions[doc["url"]] = _read_cache(path) or []
    else:
        print("  not cached. Skipped: set RUN_LLM = True to extract it (paid).")

print()
for doc in documents:
    if doc["url"] in raw_extractions:
        print("=" * 100)
        print(f"RAW ANSWER for {doc['source']} (as the Extractor returned it, before any check)")
        print("=" * 100)
        show_json(raw_extractions[doc["url"]])
        print()

## 6. Verify: every event against its source text

Done in code (`backend/app/extract.py`). An event is kept only if

- its quote is found in the source text (case, punctuation and spacing are ignored, so transcripts
  work),
- the quote names the day the event starts or ends on, in digits or spelled out (any day of a
  span of up to three days; an event that spans the whole month is exempt),
- its dates touch the month (a span reaching out of the month is cut to it).

A sky event (a new moon, a planet changing sign) has one exact day. The days the source says it is
felt ('active 3 days before, 3 days after') are kept beside it as its window, a side only when the quote names that day or counts days before or after; its dates are not checked.

A failed check drops that one event, never the whole forecast.

In [ ]:
verified = []
for doc in documents:
    if doc["url"] not in raw_extractions:
        continue
    kept, dropped = verify_answer(raw_extractions[doc["url"]], doc, month_range)
    verified.extend(kept)
    print(f"{doc['source']}: {len(kept)} kept, {len(dropped)} dropped")
    for event in kept:
        dates = event.date + (f" to {event.end_date}" if event.end_date else "")
        print(f"  kept     {dates:<24} [{event.type}] {event.event.title}  (id {event.id})")
        if event.active_from or event.active_until:
            print(
                f"             felt {event.active_from or '...'} to {event.active_until or '...'}"
            )
        show(f'"{event.event.quote}"', indent="             ")
    for reason in dropped:
        print(f"  DROPPED  {reason}")
    print()
print(f"{len(verified)} verified event(s) in total.")

## 7. Rank: what matters to you first

Done in code (`backend/app/priorities.py`, `forecast.py`). The house of an event is the one the
source states; when the source names only a zodiac sign, the whole-sign house counted from your
reading sign, and only then are the life areas of that house added to the event. An event touches a priority
through its life areas or its house. Nothing is filtered out: events outside your priorities come
last.

In [ ]:
events = rank_events(verified, priorities, reading["sign"])
for event in events:
    place = "-" if event.rank is None else str(event.rank + 1)
    house = f"house {event.house}" if event.house else "no house"
    stated = "" if event.verified.event.house or not event.house else " (computed)"
    print(
        f"  priority {place:<2} {event.verified.date}  {event.verified.event.title[:50]:<50} "
        f"{house}{stated}; {', '.join(event.life_areas)}"
    )
if not events:
    print("No verified events: the app returns an empty forecast and makes no Synthesizer call.")

## 8. Synthesize: the wording (costs money, only with `RUN_LLM = True`)

One request. The user message holds the month, the sign, your priorities text, your dated plans
and the verified events: their ids, dates, types, life areas, and the Extractor's title and advice.
**No source text and no quotes.** The Synthesizer refers to events by id and writes the theme, and
a title, summary and personal note per event. It cannot add an event or change a date.

The answer is saved under `data/notebook/` (git-ignored), so a later run with `RUN_LLM = False` can
show it again for free.

In [ ]:
synth_payload = synthesizer_payload(month, reading["sign"], priorities, upcoming_events, events)

print("SYSTEM PROMPT")
print("-" * 100)
show(SYNTHESIZER_PROMPT)
print("\nUSER MESSAGE")
print("-" * 100)
show_json(synth_payload)

event_ids = sorted(event.id for event in events)
synthesized = None
print()
if not events:
    print("Skipped: there are no verified events.")
elif RUN_LLM:
    before = len(http_log)
    started = time.time()
    try:
        synthesized = await call_nebius_chat(config, deps, SYNTHESIZER_PROMPT, synth_payload)
    except PipelineError as exc:
        print(f"The call failed: {exc.message}\n")
    calls = [entry for entry in http_log[before:] if entry["method"] == "POST"]
    print(f"{len(calls)} request(s) to {config.nebius_model} in {time.time() - started:.1f} s")
    for attempt, entry in enumerate(calls, 1):
        body = entry.get("response")
        choice = (body.get("choices") or [{}])[0] if isinstance(body, dict) else {}
        print(
            f"\nAttempt {attempt}: HTTP {entry['status']}, finish_reason={choice.get('finish_reason')}"
        )
        print("Token usage:", body.get("usage") if isinstance(body, dict) else None)
        print("RAW ANSWER")
        print("-" * 100)
        show((choice.get("message") or {}).get("content") if choice else body)
    if synthesized is not None:
        ANSWER_FILE.parent.mkdir(parents=True, exist_ok=True)
        ANSWER_FILE.write_text(
            json.dumps(
                {
                    "savedAt": time.strftime("%Y-%m-%d %H:%M"),
                    "model": config.nebius_model,
                    "sign": target.sign,
                    "month": target.label,
                    "eventIds": event_ids,
                    "priorities": synth_payload["priorities"],
                    "answer": synthesized,
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
        ANSWER_FILE.chmod(0o600)
        print(f"\nSaved to {ANSWER_FILE}")
elif ANSWER_FILE.is_file():
    saved = json.loads(ANSWER_FILE.read_text(encoding="utf-8"))
    if saved.get("eventIds") == event_ids:
        synthesized = saved["answer"]
        print(
            f"RUN_LLM is off. Showing the saved answer of {saved['savedAt']} from {saved['model']}."
        )
        if saved.get("priorities") != synth_payload["priorities"]:
            print("Note: it was made for other priorities (life context or topics changed since).")
        print("ANSWER (parsed JSON)")
        print("-" * 100)
        show_json(synthesized)
    else:
        print(
            "RUN_LLM is off, and the saved answer is for other events "
            f"({saved.get('sign')}, {saved.get('month')}). Skipped. Set RUN_LLM = True to make the call."
        )
else:
    print(
        "Skipped: RUN_LLM is off and there is no saved answer. Set RUN_LLM = True to make the call."
    )

## 9. Assemble the forecast

Done in code (`forecast.assemble_forecast`). An id that does not exist or was used before is
skipped. Of the events grouped by the Synthesizer, those with the same dates become one event; its
citations and life areas come from all of them. The others in the group stay separate. An event the Synthesizer left out is still shown,
in the source's own words. Only an answer without a usable theme fails the forecast.

In [ ]:
validated = None
if synthesized is None:
    print("Skipped: there is no Synthesizer answer.")
else:
    # Personal notes are kept only when a life context was sent, as in the app.
    assembly_errors, validated = assemble_forecast(
        synthesized, events, personal_notes=bool(life_context)
    )
    if assembly_errors:
        validated = None
        print("REJECTED. The app would answer with an error. Problems found:")
        for problem in assembly_errors:
            print("  -", problem)
    else:
        print(f"Accepted: {len(validated.events)} event(s) from {len(events)} verified one(s).\n")
        print("THEME")
        show(validated.theme)
        for event in validated.events:
            dates = event.date + (f" to {event.end_date}" if event.end_date else "")
            focus = "  * in your focus" if event.in_focus else ""
            print(f"\n{dates}  [{event.type}]  {event.title}  (id {event.id}){focus}")
            print(f"  Life areas: {', '.join(event.life_areas)}")
            details = [
                ", ".join(event.planets or []),
                event.aspect,
                event.house and f"house {event.house}",
            ]
            if any(details):
                print("  Sky:", " | ".join(str(detail) for detail in details if detail))
            for window in event.active_windows or []:
                print(
                    f"  Felt {window.start or '...'} to {window.end or '...'}: {window.source_url}"
                )
            show(event.summary, indent="  ")
            if event.personal_note:
                show(f"For you: {event.personal_note}", indent="  ")
            for url in event.source_urls:
                print(f"  Cites: {url}")

## 10. The calendar file

Built by the app from the assembled events, without the LLM.

In [ ]:
if validated is None:
    print("Skipped: there is no accepted forecast.")
else:
    print(validated.ics_content)

In [ ]:
await http.aclose()